# 06 — B6 ε-Constraint MILP

## Mục tiêu
Triển khai policy B6 đúng framing nghiên cứu:
- urgency;
- time;
- hospital capability/capacity;
- SOC/energy;
- uncertainty.

**Không dùng weighted sum tùy ý rồi gọi là B6.**

## Khuyến nghị solver
Có thể dùng `scipy.optimize.milp` (HiGHS) hoặc một MILP solver khác, nhưng phải ghi:
- solver;
- version;
- runtime;
- MIP gap;
- ε bounds;
- objective values.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import milp, LinearConstraint, Bounds

print("SciPy:", scipy.__version__)
print("milp available:", callable(milp))

## Formulation contract

Tại decision epoch, với candidate set \(C_t\):

Binary:
\[
x_c \in \{0,1\}
\]

Assignment:
\[
\sum_{c\in C_t} x_c = 1
\]

Hard filters loại trước:
- unavailable vehicle;
- incompatible hospital;
- no modeled capacity;
- SOC below reserve after mission/charger safety.

Primary objective có thể là urgency-weighted response-time proxy.

Các mục tiêu phụ được chuyển thành ε constraints, ví dụ:
\[
f_2(x)\le \epsilon_2,\quad
f_3(x)\le \epsilon_3
\]

Frontier được tạo bằng nhiều lần giải với ε khác nhau.

## Runtime candidate schema tối thiểu

```text
candidate_id
vehicle_id
hospital_id
urgency_weighted_response
transport_time
hospital_wait
energy_risk
uncertainty_risk
hard_feasible
```

In [ ]:
required_cols = [
    "candidate_id","vehicle_id","hospital_id",
    "urgency_weighted_response","transport_time",
    "hospital_wait","energy_risk","uncertainty_risk",
    "hard_feasible"
]
pd.DataFrame({"required_column": required_cols})

## Solver scaffold

Cell dưới chỉ là **contract example**. Không tạo kết quả giả nếu runtime candidate data chưa tồn tại.

In [ ]:
def solve_b6_epoch(candidates: pd.DataFrame, eps_transport=None, eps_energy=None):
    c = candidates.loc[candidates["hard_feasible"]].copy().reset_index(drop=True)
    if c.empty:
        return {"status":"NO_FEASIBLE_CANDIDATE"}

    n = len(c)
    objective = c["urgency_weighted_response"].to_numpy(float)

    # exactly one candidate
    Aeq = np.ones((1, n))
    constraints = [LinearConstraint(Aeq, lb=[1.0], ub=[1.0])]

    if eps_transport is not None:
        constraints.append(
            LinearConstraint(
                c["transport_time"].to_numpy(float)[None, :],
                lb=[-np.inf], ub=[float(eps_transport)]
            )
        )
    if eps_energy is not None:
        constraints.append(
            LinearConstraint(
                c["energy_risk"].to_numpy(float)[None, :],
                lb=[-np.inf], ub=[float(eps_energy)]
            )
        )

    res = milp(
        c=objective,
        integrality=np.ones(n),
        bounds=Bounds(np.zeros(n), np.ones(n)),
        constraints=constraints,
        options={"time_limit": 30.0}
    )

    if not res.success:
        return {"status":"SOLVER_FAIL", "message":res.message}

    idx = int(np.argmax(res.x))
    return {
        "status":"OPTIMAL_OR_ACCEPTED",
        "candidate_id": c.loc[idx, "candidate_id"],
        "objective": float(res.fun),
        "solver_message": res.message,
    }

print("B6 solver scaffold ready.")

## Gate
B6 chỉ được đưa vào 30 paired replications khi:
- candidate data là runtime state, không phải static reference snapshot;
- hard constraints đã audit;
- ε-grid được pre-specify;
- solver diagnostics được log;
- smoke test cho infeasible epoch đã pass.